In [3]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/ptls_urls.csv"
output_folder = "./database/new_ptls_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scrape_category_page(url, soup):
    """Scrape category/parts listing page (Newfind page)"""
    category_data = {}
    category_data['url'] = url
    category_data['page_type'] = 'category_listing'

    # Page metadata from container
    page_container = soup.find('div', attrs={'data-page-type': 'Newfind'})
    if page_container:
        category_data['brand'] = page_container.get('data-brand')
        category_data['model_type'] = page_container.get('data-modeltype')
    else:
        category_data['brand'] = None
        category_data['model_type'] = None

    # Meta information
    canonical = soup.find('link', rel='canonical')
    category_data['canonical_url'] = canonical.get('href') if canonical else None
    
    meta_desc = soup.find('meta', attrs={'name': 'description'})
    category_data['meta_description'] = meta_desc.get('content') if meta_desc else None

    # Page title
    title = soup.find('h1', class_='title-main')
    category_data['page_title'] = title.text.strip() if title else None

    # Brand image
    brand_img_div = soup.find('div', class_='nf__brand')
    if brand_img_div:
        img_tag = brand_img_div.find('img')
        category_data['brand_image'] = {
            'url': img_tag.get('src') if img_tag else None,
            'alt': img_tag.get('alt') if img_tag else None
        }
    else:
        category_data['brand_image'] = None

    # Search box information
    searchbox = soup.find('div', class_='searchbox')
    if searchbox:
        search_input = searchbox.find('input', id='searchboxInput')
        category_data['search_box'] = {
            'placeholder': search_input.get('placeholder') if search_input else None,
            'has_search': True
        }
    else:
        category_data['search_box'] = None

    # Parts listing
    parts = []
    part_elements = soup.find_all('div', class_='nf__part')
    
    for part_elem in part_elements:
        part_data = {}
        
        # Part image
        img_elem = part_elem.find('img', class_='b-lazy')
        if img_elem:
            part_data['image'] = {
                'url': img_elem.get('data-src'),
                'title': img_elem.get('title'),
                'alt': img_elem.get('alt')
            }
        else:
            part_data['image'] = None

        # Part title/name and URL
        title_elem = part_elem.find('a', class_='nf__part__detail__title')
        if title_elem:
            span = title_elem.find('span')
            part_data['name'] = span.text.strip() if span else None
            part_data['url'] = 'https://www.partselect.com' + title_elem.get('href') if title_elem.get('href') else None
        else:
            part_data['name'] = None
            part_data['url'] = None

        # Rating and reviews
        rating_elem = part_elem.find('a', class_='nf__part__detail__rating')
        if rating_elem:
            rating_div = rating_elem.find('div', class_='rating')
            if rating_div:
                # Extract rating from star width percentage
                stars_upper = rating_div.find('div', class_='rating__stars__upper')
                if stars_upper:
                    style = stars_upper.get('style', '')
                    if 'width:' in style:
                        percentage = style.split('width:')[1].split('%')[0].strip()
                        try:
                            part_data['rating'] = round(float(percentage) / 20, 2)  # Convert to 5-star scale
                        except:
                            part_data['rating'] = None
                    else:
                        part_data['rating'] = None
                else:
                    part_data['rating'] = None
                
                # Review count
                review_count = rating_div.find('span', class_='rating__count')
                part_data['review_count'] = review_count.text.strip() if review_count else None
            else:
                part_data['rating'] = None
                part_data['review_count'] = None
        else:
            part_data['rating'] = None
            part_data['review_count'] = None

        # Part numbers
        part_numbers = part_elem.find_all('div', class_='nf__part__detail__part-number')
        if len(part_numbers) >= 1:
            ps_num = part_numbers[0].find('strong')
            part_data['partselect_number'] = ps_num.text.strip() if ps_num else None
        else:
            part_data['partselect_number'] = None
            
        if len(part_numbers) >= 2:
            mfr_num = part_numbers[1].find('strong')
            part_data['manufacturer_part_number'] = mfr_num.text.strip() if mfr_num else None
        else:
            part_data['manufacturer_part_number'] = None

        # Description (text content after part numbers)
        desc_paragraphs = []
        for elem in part_elem.find_all('div', class_='nf__part__detail'):
            # Get all text nodes that aren't in specific child divs
            for content in elem.children:
                if isinstance(content, str):
                    text = content.strip()
                    if text and len(text) > 20:  # Filter out empty or very short strings
                        desc_paragraphs.append(text)
        
        part_data['description'] = ' '.join(desc_paragraphs) if desc_paragraphs else None

        # Price information
        price_div = part_elem.find('div', class_='nf__part__left-col__basic-info__price')
        if price_div:
            price_elem = price_div.find('div', class_='price')
            if price_elem:
                price_text = price_elem.get_text(strip=True).replace('$', '').strip()
                part_data['price'] = price_text if price_text else None
            else:
                part_data['price'] = None

            # Original price (if on sale)
            original_price = price_div.find('div', class_='original-price')
            part_data['original_price'] = original_price.get_text(strip=True).replace('$', '').strip() if original_price else None

            # Discount badge
            discount_badge = price_div.find('div', class_='price__discount-badge')
            if discount_badge:
                discount_text = discount_badge.find('span')
                part_data['discount'] = discount_text.text.strip() if discount_text else None
            else:
                part_data['discount'] = None
        else:
            part_data['price'] = None
            part_data['original_price'] = None
            part_data['discount'] = None

        # Stock status
        stock_div = part_elem.find('div', class_='nf__part__left-col__basic-info__stock')
        if stock_div:
            stock_elem = stock_div.find('span')
            part_data['stock_status'] = stock_elem.text.strip() if stock_elem else None
        else:
            part_data['stock_status'] = None

        # Add to cart button data
        cart_button = part_elem.find('button', class_='js-addToCart')
        if cart_button:
            data_args = cart_button.get('data-args', '')
            part_data['cart_data'] = data_args
        else:
            part_data['cart_data'] = None

        # Symptoms fixed
        symptoms = []
        symptoms_section = part_elem.find('div', class_='nf__part__detail__symptoms')
        if symptoms_section:
            symptom_list = symptoms_section.find('ul')
            if symptom_list:
                for li in symptom_list.find_all('li'):
                    # Skip the "See more..." link
                    if not li.find('a'):
                        symptoms.append(li.text.strip())
        part_data['fixes_symptoms'] = symptoms if symptoms else None

        # Installation instructions/repair story
        instruction_section = part_elem.find('div', class_='nf__part__detail__instruction')
        if instruction_section:
            creator = instruction_section.find('div', class_='nf__part__detail__instruction__creator')
            quote_div = instruction_section.find('div', class_='nf__part__detail__instruction__quote')
            
            if quote_div:
                quote_title = quote_div.find('div', class_='bold')
                quote_text = quote_div.find('span', class_='d-block')
                read_more = quote_div.find('a')
                
                part_data['installation_story'] = {
                    'author': creator.text.strip() if creator else None,
                    'title': quote_title.text.strip() if quote_title else None,
                    'description': quote_text.text.strip() if quote_text else None,
                    'read_more_url': 'https://www.partselect.com' + read_more.get('href') if read_more else None
                }
            else:
                part_data['installation_story'] = None
        else:
            part_data['installation_story'] = None

        parts.append(part_data)
    
    category_data['parts'] = parts

    # Section title for parts
    section_title = soup.find('h2', class_='section-title')
    category_data['parts_section_title'] = section_title.text.strip() if section_title else None

    # Related parts categories
    related_parts = []
    related_section = soup.find('h2', id='ShopByPartType')
    if related_section:
        links_ul = related_section.find_next('ul', class_='nf__links')
        if links_ul:
            for li in links_ul.find_all('li'):
                link = li.find('a')
                if link:
                    related_parts.append({
                        'name': link.text.strip(),
                        'url': 'https://www.partselect.com' + link.get('href') if link.get('href') else None
                    })
    
    category_data['related_parts_categories'] = related_parts

    # Filters section
    filters_div = soup.find('div', class_='nf__filters')
    if filters_div:
        filter_links = []
        for link in filters_div.find_all('a'):
            filter_links.append({
                'text': link.text.strip(),
                'href': link.get('href')
            })
        category_data['filters'] = filter_links
    else:
        category_data['filters'] = None

    # OEM badge info
    oem_badge = soup.find('a', class_='oem-badge')
    if oem_badge:
        img = oem_badge.find('img')
        p = oem_badge.find('p')
        category_data['oem_badge'] = {
            'image': img.get('src') if img else None,
            'text': p.text.strip() if p else None,
            'url': oem_badge.get('href')
        }
    else:
        category_data['oem_badge'] = None

    return category_data

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(urls)} URLs to scrape")

for idx, url in enumerate(urls, 1):
    try:
        print(f"\n[{idx}/{len(urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for page to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Scrape the category page
        data = scrape_category_page(url, soup)

        # Generate filename from brand and category
        parts = []
        if data.get('brand'):
            parts.append(data['brand'])
        if data.get('model_type'):
            parts.append(data['model_type'])
        
        # Try to get category from page title
        if data.get('page_title'):
            title_parts = data['page_title'].split()
            if len(title_parts) > 2:
                parts.append(title_parts[-3] + '-' + title_parts[-2])  # e.g., "Caps-and-Lids"
        
        filename = '-'.join(parts) if parts else url.split('/')[-1].replace('.htm', '')
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        base_filename = filename.replace(".json", "")
        filepath = os.path.join(output_folder, filename)

        # Handle duplicates by adding _1, _2, etc.
        counter = 1
        while os.path.exists(filepath):
            new_filename = f"{base_filename}_{counter}.json"
            filepath = os.path.join(output_folder, new_filename)
            counter += 1

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {os.path.basename(filepath)}")
        print(f"    - Brand: {data.get('brand', 'N/A')}")
        print(f"    - Model Type: {data.get('model_type', 'N/A')}")
        print(f"    - Parts listed: {len(data.get('parts', []))}")
        print(f"    - Related categories: {len(data.get('related_parts_categories', []))}")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        import traceback
        traceback.print_exc()
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped data saved to:", output_folder)

Found 1540 URLs to scrape

[1/1540] Processing: https://www.partselect.com/Admiral-Dishwasher-Brackets-and-Flanges.htm
  ✓ Saved Admiral-Dishwasher-Brackets-and
    - Brand: Admiral
    - Model Type: Dishwasher
    - Parts listed: 10
    - Related categories: 8

[2/1540] Processing: https://www.partselect.com/Admiral-Dishwasher-Caps-and-Lids.htm


KeyboardInterrupt: 

In [4]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

csv_file = "./filtered_urls/ptls_urls.csv"
output_folder = "./database/new_ptls_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options
def get_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    driver = webdriver.Chrome(options=options)
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    return driver

# Function to scrape a single URL
def scrape_url(url):
    driver = get_driver()
    try:
        print(f"Processing: {url}")
        driver.get(url)

        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")

        time.sleep(random.uniform(2, 4))
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        data = scrape_category_page(url, soup)

        # Filename logic
        parts = []
        if data.get('brand'):
            parts.append(data['brand'])
        if data.get('model_type'):
            parts.append(data['model_type'])
        if data.get('page_title'):
            title_parts = data['page_title'].split()
            if len(title_parts) > 2:
                parts.append(title_parts[-3] + '-' + title_parts[-2])
        filename = '-'.join(parts) if parts else url.split('/')[-1].replace('.htm', '')
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        base_filename = filename.replace(".json", "")
        filepath = os.path.join(output_folder, filename)
        counter = 1
        while os.path.exists(filepath):
            filepath = os.path.join(output_folder, f"{base_filename}_{counter}.json")
            counter += 1

        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {os.path.basename(filepath)}")
        return url, True
    except Exception as e:
        print(f"  ✗ Failed for {url}: {e}")
        return url, False
    finally:
        driver.quit()

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    exit(1)

# Run threads
max_threads = 4  # adjust based on CPU/memory
with ThreadPoolExecutor(max_threads) as executor:
    futures = [executor.submit(scrape_url, url) for url in urls]
    for future in as_completed(futures):
        url, success = future.result()
        if success:
            print(f"[DONE] {url}")
        else:
            print(f"[FAILED] {url}")

print("\n✓ All done! Scraped data saved to:", output_folder)


Processing: https://www.partselect.com/Admiral-Dishwasher-Brackets-and-Flanges.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Dispensers.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Caps-and-Lids.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Dishracks.htm
  ✓ Saved Admiral-Dishwasher-Caps-and
  ✓ Saved Admiral-Dishwasher-Admiral-Dishwasher
  ✓ Saved Admiral-Dishwasher-Admiral-Dishwasher_1.json
  ✓ Saved Admiral-Dishwasher-Brackets-and_1.json
[DONE] https://www.partselect.com/Admiral-Dishwasher-Caps-and-Lids.htm
[DONE] https://www.partselect.com/Admiral-Dishwasher-Dishracks.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Hardware.htm
[DONE] https://www.partselect.com/Admiral-Dishwasher-Brackets-and-Flanges.htm
[DONE] https://www.partselect.com/Admiral-Dishwasher-Dispensers.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Hinges.htm
Processing: https://www.partselect.com/Admiral-Dishwasher-Hoses-and-Tubes.htm
Proc

[DONE] https://www.partselect.com/Amana-Dishwasher-Dishracks.htm
  ✓ Saved Amana-Dishwasher-Amana-Dishwasher_1.json
Processing: https://www.partselect.com/Amana-Dishwasher-Hardware.htm
  ✓ Saved Amana-Dishwasher-Amana-Dishwasher_2.json
[DONE] https://www.partselect.com/Amana-Dishwasher-Dispensers.htm
Processing: https://www.partselect.com/Amana-Dishwasher-Hinges.htm
[DONE] https://www.partselect.com/Amana-Dishwasher-Doors.htm
  ✓ Saved Amana-Dishwasher-Amana-Dishwasher_3.json
Processing: https://www.partselect.com/Amana-Dishwasher-Hoses-and-Tubes.htm
[DONE] https://www.partselect.com/Amana-Dishwasher-Handles.htm
  ✓ Saved Amana-Dishwasher-Amana-Dishwasher_4.json
Processing: https://www.partselect.com/Amana-Dishwasher-Latches.htm
[DONE] https://www.partselect.com/Amana-Dishwasher-Hardware.htm
  ✓ Saved Amana-Dishwasher-Amana-Dishwasher_5.json
Processing: https://www.partselect.com/Amana-Dishwasher-Panels.htm
[DONE] https://www.partselect.com/Amana-Dishwasher-Hinges.htm
  ✓ Saved Amana-D

[DONE] https://www.partselect.com/Amana-Refrigerator-Seals-and-Gaskets.htm
  ✓ Saved Amana-Refrigerator-Amana-Refrigerator_13.json
Processing: https://www.partselect.com/Amana-Refrigerator-Trim.htm
[DONE] https://www.partselect.com/Amana-Refrigerator-Switches.htm
Processing: https://www.partselect.com/Amana-Refrigerator-Trims.htm
[DONE] https://www.partselect.com/Amana-Refrigerator-Thermostats.htm
  ✓ Saved Amana-Refrigerator-Trays-and
Processing: https://www.partselect.com/Amana-Refrigerator-Valves.htm
[DONE] https://www.partselect.com/Amana-Refrigerator-Trays-and-Shelves.htm
Processing: https://www.partselect.com/Amana-Refrigerator-Wheels-and-Rollers.htm
  ✓ Saved Amana-Refrigerator-Amana-Refrigerator_14.json
  ✓ Saved Page-Not_3.json
[DONE] https://www.partselect.com/Amana-Refrigerator-Trim.htm
  ✓ Saved Amana-Refrigerator-Amana-Refrigerator_15.json
[DONE] https://www.partselect.com/Amana-Refrigerator-Trims.htm
Processing: https://www.partselect.com/Amana-Refrigerator-Wire-Plugs-and

[DONE] https://www.partselect.com/Bosch-Refrigerator-Hardware.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Lights-and-Bulbs.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Motors.htm
  ✓ Saved Bosch-Refrigerator-Refrigerator-Ice
  ✓ Saved Bosch-Refrigerator-Hoses-and
[DONE] https://www.partselect.com/Bosch-Refrigerator-Ice-Makers.htm
[DONE] https://www.partselect.com/Bosch-Refrigerator-Hoses-and-Tubes.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Panels.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Seals-and-Gaskets.htm
  ✓ Saved Bosch-Refrigerator-Lights-and
  ✓ Saved Bosch-Refrigerator-Bosch-Refrigerator_7.json
[DONE] https://www.partselect.com/Bosch-Refrigerator-Lights-and-Bulbs.htm
[DONE] https://www.partselect.com/Bosch-Refrigerator-Motors.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Sensors.htm
Processing: https://www.partselect.com/Bosch-Refrigerator-Switches.htm
  ✓ Saved Bosch-Refrigerator-Seals-an

Processing: https://www.partselect.com/Crosley-Refrigerator-Drip-Bowls.htm
[DONE] https://www.partselect.com/Crosley-Refrigerator-Compressors.htm
[DONE] https://www.partselect.com/Crosley-Refrigerator-Doors.htm
Processing: https://www.partselect.com/Crosley-Refrigerator-Ducts-and-Vents.htm
Processing: https://www.partselect.com/Crosley-Refrigerator-Elements-and-Burners.htm
  ✓ Saved Crosley-Refrigerator-Drawers-and
[DONE] https://www.partselect.com/Crosley-Refrigerator-Drawers-and-Glides.htm
Processing: https://www.partselect.com/Crosley-Refrigerator-Fans-and-Blowers.htm
  ✓ Saved Crosley-Refrigerator-Refrigerator-Drip
  ✓ Saved Crosley-Refrigerator-Ducts-and
  ✓ Saved Crosley-Refrigerator-Elements-and
[DONE] https://www.partselect.com/Crosley-Refrigerator-Drip-Bowls.htm
[DONE] https://www.partselect.com/Crosley-Refrigerator-Ducts-and-Vents.htm
Processing: https://www.partselect.com/Crosley-Refrigerator-Filters.htm
[DONE] https://www.partselect.com/Crosley-Refrigerator-Elements-and-Bur

...

[DONE] https://www.partselect.com/Thermador-Refrigerator-Filters.htm
Processing: https://www.partselect.com/Thermador-Refrigerator-Hinges.htm
  ✓ Saved Thermador-Refrigerator-Thermador-Refrigerator_3.json
Processing: https://www.partselect.com/Thermador-Refrigerator-Hoses-and-Tubes.htm
  ✓ Saved Thermador-Refrigerator-Thermador-Refrigerator_4.json
[DONE] https://www.partselect.com/Thermador-Refrigerator-Handles.htm
Processing: https://www.partselect.com/Thermador-Refrigerator-Ice-Makers.htm
[DONE] https://www.partselect.com/Thermador-Refrigerator-Hardware.htm
Processing: https://www.partselect.com/Thermador-Refrigerator-Latches.htm
  ✓ Saved Thermador-Refrigerator-Hoses-and
  ✓ Saved Thermador-Refrigerator-Thermador-Refrigerator_5.json
[DONE] https://www.partselect.com/Thermador-Refrigerator-Hoses-and-Tubes.htm
  ✓ Saved Thermador-Refrigerator-Refrigerator-Ice
[DONE] https://www.partselect.com/Thermador-Refrigerator-Hinges.htm
Processing: https://www.partselect.com/Thermador-Refrigerat

  ✓ Saved Uni-Refrigerator-Uni-Refrigerator_9.json
  ✓ Saved Uni-Refrigerator-Lights-and
[DONE] https://www.partselect.com/Uni-Refrigerator-Panels.htm
[DONE] https://www.partselect.com/Uni-Refrigerator-Lights-and-Bulbs.htm
Processing: https://www.partselect.com/Uni-Refrigerator-Switches.htm
Processing: https://www.partselect.com/Uni-Refrigerator-Thermostats.htm
  ✓ Saved Uni-Refrigerator-Seals-and
  ✓ Saved Uni-Refrigerator-and-Shock
[DONE] https://www.partselect.com/Uni-Refrigerator-Springs-and-Shock-Absorbers.htm
[DONE] https://www.partselect.com/Uni-Refrigerator-Seals-and-Gaskets.htm
Processing: https://www.partselect.com/Uni-Refrigerator-Trays-and-Shelves.htm
Processing: https://www.partselect.com/Uni-Refrigerator-Trim.htm
  ✓ Saved Uni-Refrigerator-Uni-Refrigerator_10.json
  ✓ Saved Uni-Refrigerator-Uni-Refrigerator_11.json
[DONE] https://www.partselect.com/Uni-Refrigerator-Switches.htm
Processing: https://www.partselect.com/Uni-Refrigerator-Trims.htm
[DONE] https://www.partselect

Processing: https://www.partselect.com/Whirlpool-Dishwasher-Brackets-and-Flanges.htm
  ✓ Saved Westinghouse-Refrigerator-Westinghouse-Refrigerator_14.json
[DONE] https://www.partselect.com/Westinghouse-Refrigerator-Trims.htm
Processing: https://www.partselect.com/Whirlpool-Dishwasher-Caps-and-Lids.htm
  ✓ Saved Westinghouse-Refrigerator-Wheels-and
  ✓ Saved Westinghouse-Refrigerator-Plugs-and
[DONE] https://www.partselect.com/Westinghouse-Refrigerator-Wheels-and-Rollers.htm
  ✓ Saved Whirlpool-Dishwasher-Brackets-and
Processing: https://www.partselect.com/Whirlpool-Dishwasher-Circuit-Boards-and-Touch-Pads.htm
[DONE] https://www.partselect.com/Westinghouse-Refrigerator-Wire-Plugs-and-Connectors.htm
  ✓ Saved Whirlpool-Dishwasher-Caps-and
Processing: https://www.partselect.com/Whirlpool-Dishwasher-Deflectors-and-Chutes.htm
[DONE] https://www.partselect.com/Whirlpool-Dishwasher-Brackets-and-Flanges.htm
[DONE] https://www.partselect.com/Whirlpool-Dishwasher-Caps-and-Lids.htm
Processing: ht

Processing: https://www.partselect.com/Whirlpool-Refrigerator-Compressors.htm
  ✓ Saved Whirlpool-Refrigerator-Brackets-and
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Brackets-and-Flanges.htm
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Deflectors-and-Chutes.htm
  ✓ Saved Whirlpool-Refrigerator-Caps-and
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Caps-and-Lids.htm
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Dispensers.htm
  ✓ Saved Whirlpool-Refrigerator-and-Touch
  ✓ Saved Whirlpool-Refrigerator-Whirlpool-Refrigerator_1.json
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Circuit-Boards-and-Touch-Pads.htm
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Compressors.htm
  ✓ Saved Whirlpool-Refrigerator-Deflectors-and
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Doors.htm
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Drawers-and-Glides.htm
[DONE] https://www.partselect.com/Whi

Processing: https://www.partselect.com/Whirlpool-Refrigerator-Trims.htm
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Trays-and-Shelves.htm
  ✓ Saved Whirlpool-Refrigerator-Whirlpool-Refrigerator_19.json
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Valves.htm
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Wheels-and-Rollers.htm
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Trim.htm
Processing: https://www.partselect.com/Whirlpool-Refrigerator-Wire-Plugs-and-Connectors.htm
  ✓ Saved Page-Not_57.json
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Trims.htm
  ✓ Saved Whirlpool-Refrigerator-Whirlpool-Refrigerator_20.json
  ✓ Saved Whirlpool-Refrigerator-Wheels-and
Processing: https://www.partselect.com/White-Westinghouse-Dishwasher-Hardware.htm
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Valves.htm
[DONE] https://www.partselect.com/Whirlpool-Refrigerator-Wheels-and-Rollers.htm
Processing: https://www.partselect.co